# Middleware in LangChain

Let's start with the core agent loop, then add middleware as hooks around that loop.


In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain langchain-core langchain-openai langgraph python-dotenv

In [ ]:
import getpass
import os

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

In [ ]:
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "middleware"

## The Core Agent Loop

Before middleware matters, let's understand the loop it wraps.

A LangChain agent created with `create_agent()` repeatedly does this:

1. Read the conversation so far.
2. Call the model to decide the next step.
3. If the model asks for a tool, run that tool.
4. Add the tool result back into the conversation.
5. Repeat until the model gives a final answer.

In our support desk analogy, the representative listens to the customer, decides whether to check an internal system, reads the result, and then either checks another system or replies.

### Support Agent With One Tool

This first agent can look up an order status. There is no middleware yet.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a customer's order status by order ID."""
    orders = {
        "A100": "shipped yesterday and arrives tomorrow",
        "B200": "still being packed in the warehouse",
    }
    return orders.get(order_id, "order not found")


support_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[lookup_order],
    system_prompt="You are a concise customer support agent. Use tools when needed.",
)

result = support_agent.invoke(
    {"messages": [{"role": "user", "content": "Can you check order A100?"}]}
)

print(result["messages"][-1].content)